# Symulator tomografu - GUI w notatniku

Ten notatnik uruchamia interfejs oparty o `ipywidgets`.

## Co potrafi GUI
- wybór obrazu `.jpg` z katalogu `tomograf-obrazy`
- konfiguracja parametrów rekonstrukcji
- podgląd: obraz oryginalny, sinogram i rekonstrukcja
- wykres historii MSE
- zapis wyniku do DICOM
- batch dla wszystkich obrazów `.jpg`

Uruchom komórki od góry do dołu i na końcu klikaj przyciski w panelu.

## 1) Importy i konfiguracja
Poniższa komórka ładuje moduły projektu i przygotowuje foldery wejścia/wyjścia.

In [9]:
# Importy i konfiguracja

from pathlib import Path
import datetime
import time

import imageio.v2 as iio
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import clear_output, display
from skimage.transform import resize

import filter
import save_dicom
import utils

BASE_DIR = Path('.').resolve()
IMAGES_DIR = BASE_DIR / 'tomograf-obrazy'
DICOM_DIR = BASE_DIR / 'dicom'
DICOM_DIR.mkdir(parents=True, exist_ok=True)

APP_STATE = {
    'last_original': None,
    'last_sinogram': None,
    'last_raw_reconstruction': None,
    'last_filtered_reconstruction': None,
    'last_mse_raw': None,
    'last_mse_filtered': None,
    'last_filename': None,
    'last_params': None,
}

print(f'Katalog obrazow: {IMAGES_DIR}')
print(f'Katalog DICOM:   {DICOM_DIR}')

Katalog obrazow: /Users/krzysztofzaporowski/Desktop/Studia/InformatykaWMedycynie/tomograf/tomograf-obrazy
Katalog DICOM:   /Users/krzysztofzaporowski/Desktop/Studia/InformatykaWMedycynie/tomograf/dicom


In [10]:
# Helpery backendowe wykorzystywane przez GUI

def available_image_files():
    if not IMAGES_DIR.exists():
        return []
    return sorted([p.name for p in IMAGES_DIR.glob('*.jpg')])


def load_grayscale_image(filename, image_size):
    path = IMAGES_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f'Nie znaleziono pliku: {path}')

    image = iio.imread(path.as_posix(), mode='L')
    image = resize(image, (image_size, image_size))
    return image.astype(np.float64)


def scanner_preview(detectors, spread, image_size=100, alpha_deg=0):
    center = image_size / 2
    radius = image_size * 0.7

    emitter, detector_points = utils.wyznacz_pozycje_czujnikow(
        alpha_deg,
        detectors,
        spread,
        radius,
        center,
        center,
    )

    plt.figure(figsize=(6, 6))
    plt.plot(center, center, 'k+', markersize=15, label='Srodek obrotu')
    plt.plot(emitter[0], emitter[1], 'ro', markersize=10, label='Emiter (E)')

    for det in detector_points:
        plt.plot(det[0], det[1], 'bo', markersize=5)
        plt.plot([emitter[0], det[0]], [emitter[1], det[1]], 'gray', alpha=0.3, linestyle='--')

    plt.xlim(-20, image_size + 20)
    plt.ylim(-20, image_size + 20)
    plt.title(f'Uklad tomografu: Alfa={alpha_deg} stopni, n={detectors}, rozpietosc={spread} stopni')
    plt.legend()
    plt.grid(True)
    plt.show()


def compute_sinogram_and_reconstruction(filename, image_size, detectors, scans, spread):
    image = load_grayscale_image(filename, image_size)
    wymiar_y, wymiar_x = image.shape

    sinogram = utils.stworz_sinogram(
        wymiar_x,
        wymiar_y,
        detectors,
        scans,
        spread,
        image,
    )

    reconstruction, mse_history = utils.rekonstrukcja_obrazu(
        wymiar_x,
        wymiar_y,
        detectors,
        scans,
        spread,
        sinogram,
        image,
    )

    return image, sinogram, reconstruction, mse_history


def compute_filtered_reconstruction(image, sinogram, detectors, scans, spread, filter_method='fft'):
    wymiar_y, wymiar_x = image.shape
    sinogram_filtrowany = filter.filtruj_sinogram(sinogram, metoda=filter_method)

    rekonstrukcja_filtrowana, mse_historia_filtrowana = utils.rekonstrukcja_obrazu(
        wymiar_x,
        wymiar_y,
        detectors,
        scans,
        spread,
        sinogram_filtrowany,
        image,
    )

    return sinogram_filtrowany, rekonstrukcja_filtrowana, mse_historia_filtrowana


def plot_single_run(image, sinogram, reconstruction):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    ax1.imshow(image, cmap='gray')
    ax1.set_title('Oryginalny obraz')

    ax2.imshow(sinogram, cmap='gray', aspect='auto')
    ax2.set_title('Wygenerowany Sinogram')
    ax2.set_xlabel('Kat obrotu (skan)')
    ax2.set_ylabel('Numer detektora')
    plt.show()

    plt.imshow(reconstruction, cmap='gray')
    plt.title('Rekonstruowany obraz')
    plt.axis('off')
    plt.show()


def plot_filtered_run(image, sinogram_filtrowany, rekonstrukcja_filtrowana):
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))
    ax1.imshow(sinogram_filtrowany, cmap='gray')
    ax1.set_title('Sinogram po filtracji')

    ax2.imshow(rekonstrukcja_filtrowana, cmap='gray')
    ax2.set_title('Rekonstruowany obraz po filtracji')

    ax3.imshow(image, cmap='gray')
    ax3.set_title('Oryginalny obraz')
    plt.show()


def save_filtered_to_dicom(first_name, last_name, pesel, comment):
    reconstruction = APP_STATE.get('last_filtered_reconstruction')
    if reconstruction is None:
        raise RuntimeError('Najpierw uruchom rekonstrukcje z wlaczona filtracja.')

    data_badania = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    nazwa_pliku = (
        DICOM_DIR
        / f"rekonstrukcja_{first_name}_{last_name}_{data_badania.replace(':', '-')}.dcm"
    )

    save_dicom.zapisz_dicom(
        reconstruction,
        nazwa_pliku.as_posix(),
        (first_name, last_name, pesel),
        data_badania,
        comment,
    )

    return nazwa_pliku


def run_batch_full(image_size, detectors, scans, spread, filter_method='fft'):
    files = available_image_files()
    if not files:
        raise RuntimeError('Brak plikow .jpg w katalogu tomograf-obrazy.')

    summary = []
    for file_name in files:
        image, sinogram, rekonstrukcja, mse_historia = compute_sinogram_and_reconstruction(
            file_name,
            image_size,
            detectors,
            scans,
            spread,
        )

        sinogram_filtrowany, rekonstrukcja_filtrowana, mse_historia_filtrowana = compute_filtered_reconstruction(
            image,
            sinogram,
            detectors,
            scans,
            spread,
            filter_method=filter_method,
        )

        utils.pokaz_historię_mse(mse_historia)
        utils.pokaz_historię_mse(mse_historia_filtrowana)

        fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))
        ax1.imshow(image, cmap='gray')
        ax1.set_title('Oryginalny obraz')
        ax2.imshow(rekonstrukcja, cmap='gray')
        ax2.set_title('Rekonstruowany obraz')
        ax3.imshow(rekonstrukcja_filtrowana, cmap='gray')
        ax3.set_title('Rekonstruowany obraz po filtracji')
        plt.show()

        mse_raw = mse_historia[-1] if mse_historia else float('nan')
        mse_filt = mse_historia_filtrowana[-1] if mse_historia_filtrowana else float('nan')
        summary.append((file_name, mse_raw, mse_filt))

    return summary

## 2) Panel GUI
Poniższa komórka buduje interfejs i podłącza akcje:
- `Uruchom rekonstrukcję`
- `Zapisz aktualny wynik do DICOM`
- `Batch dla wszystkich obrazów`

In [11]:
def build_gui():
    image_files = available_image_files()

    image_dropdown = widgets.Dropdown(
        options=image_files,
        description='Plik:',
        value=image_files[0] if image_files else None,
        layout=widgets.Layout(width='430px'),
    )
    image_size_slider = widgets.IntSlider(value=100, min=64, max=256, step=4, description='Rozmiar:')
    detectors_slider = widgets.IntSlider(value=180, min=30, max=360, step=10, description='Detektory:')
    scans_slider = widgets.IntSlider(value=180, min=30, max=360, step=10, description='Skanow:')
    spread_slider = widgets.IntSlider(value=180, min=30, max=270, step=5, description='Rozpietosc:')
    use_filter_checkbox = widgets.Checkbox(value=True, description='Pokaz filtracje')
    filter_method_dropdown = widgets.Dropdown(
        options=[('Splot (Ram-Lak)', 'convolve'), ('Transformata Fouriera', 'fft')],
        value='convolve',
        description='Filtr:',
    )

    preview_button = widgets.Button(description='Podglad ukladu czujnikow')
    run_button = widgets.Button(description='Uruchom rekonstrukcje', button_style='primary')
    batch_button = widgets.Button(description='Batch dla wszystkich obrazow', button_style='info')
    dicom_button = widgets.Button(description='Zapisz DICOM (wynik filtrowany)', button_style='success')

    patient_first = widgets.Text(value='Jan', description='Imie:')
    patient_last = widgets.Text(value='Kowalski', description='Nazwisko:')
    patient_pesel = widgets.Text(value='98051599415', description='PESEL:')
    patient_comment = widgets.Text(value='Rekonstrukcja tomograficzna z filtrowaniem', description='Komentarz:')

    status_html = widgets.HTML(value='<b>Status:</b> gotowy')
    output = widgets.Output()

    def update_status(message, level='info'):
        colors = {'info': '#1f6feb', 'ok': '#1a7f37', 'warn': '#9a6700', 'error': '#cf222e'}
        status_html.value = f"<b>Status:</b> <span style='color:{colors.get(level, '#1f6feb')}'>{message}</span>"

    def on_preview_clicked(_):
        with output:
            clear_output(wait=True)
            scanner_preview(
                detectors=detectors_slider.value,
                spread=spread_slider.value,
                image_size=image_size_slider.value,
                alpha_deg=0,
            )
        update_status('Wyswietlono podglad ukladu czujnikow.', 'ok')

    def on_run_clicked(_):
        with output:
            clear_output(wait=True)
            if not image_dropdown.value:
                update_status('Brak plikow .jpg do przetworzenia.', 'error')
                return

            start = time.perf_counter()
            try:
                image, sinogram, rekonstrukcja, mse_historia = compute_sinogram_and_reconstruction(
                    filename=image_dropdown.value,
                    image_size=image_size_slider.value,
                    detectors=detectors_slider.value,
                    scans=scans_slider.value,
                    spread=spread_slider.value,
                )
                plot_single_run(image, sinogram, rekonstrukcja)
                utils.pokaz_historię_mse(mse_historia)

                APP_STATE['last_original'] = image
                APP_STATE['last_sinogram'] = sinogram
                APP_STATE['last_raw_reconstruction'] = rekonstrukcja
                APP_STATE['last_mse_raw'] = mse_historia
                APP_STATE['last_filename'] = image_dropdown.value
                APP_STATE['last_params'] = {
                    'image_size': image_size_slider.value,
                    'detectors': detectors_slider.value,
                    'scans': scans_slider.value,
                    'spread': spread_slider.value,
                    'filter_method': filter_method_dropdown.value,
                }

                if use_filter_checkbox.value:
                    sinogram_filtrowany, rekonstrukcja_filtrowana, mse_historia_filtrowana = compute_filtered_reconstruction(
                        image,
                        sinogram,
                        detectors_slider.value,
                        scans_slider.value,
                        spread_slider.value,
                        filter_method=filter_method_dropdown.value,
                    )
                    plot_filtered_run(image, sinogram_filtrowany, rekonstrukcja_filtrowana)
                    utils.pokaz_historię_mse(mse_historia_filtrowana)
                    APP_STATE['last_filtered_reconstruction'] = rekonstrukcja_filtrowana
                    APP_STATE['last_mse_filtered'] = mse_historia_filtrowana
                else:
                    APP_STATE['last_filtered_reconstruction'] = None
                    APP_STATE['last_mse_filtered'] = None

                elapsed = time.perf_counter() - start
                update_status(
                    f"Gotowe w {elapsed:.2f} s. Plik: {image_dropdown.value}. Filtr: {filter_method_dropdown.value}",
                    'ok',
                )
            except Exception as exc:
                update_status(f'Blad: {exc}', 'error')

    def on_save_dicom_clicked(_):
        with output:
            try:
                filename = save_filtered_to_dicom(
                    patient_first.value.strip(),
                    patient_last.value.strip(),
                    patient_pesel.value.strip(),
                    patient_comment.value.strip(),
                )
                print(f'Zapisano plik: {filename}')
                update_status(f'Zapisano DICOM: {filename.name}', 'ok')
            except Exception as exc:
                update_status(f'Nie udalo sie zapisac DICOM: {exc}', 'error')

    def on_batch_clicked(_):
        with output:
            clear_output(wait=True)
            start = time.perf_counter()
            try:
                summary = run_batch_full(
                    image_size=image_size_slider.value,
                    detectors=detectors_slider.value,
                    scans=scans_slider.value,
                    spread=spread_slider.value,
                    filter_method=filter_method_dropdown.value,
                )
                print('Podsumowanie batch (plik -> mse_surowe | mse_filtrowane):')
                for file_name, mse_raw, mse_filt in summary:
                    print(f' - {file_name}: {mse_raw:.6f} | {mse_filt:.6f}')
                elapsed = time.perf_counter() - start
                update_status(
                    f"Batch zakonczony w {elapsed:.2f} s. Filtr: {filter_method_dropdown.value}",
                    'ok',
                )
            except Exception as exc:
                update_status(f'Batch przerwany: {exc}', 'error')

    preview_button.on_click(on_preview_clicked)
    run_button.on_click(on_run_clicked)
    dicom_button.on_click(on_save_dicom_clicked)
    batch_button.on_click(on_batch_clicked)

    params_box = widgets.VBox([
        image_dropdown,
        image_size_slider,
        detectors_slider,
        scans_slider,
        spread_slider,
        use_filter_checkbox,
        filter_method_dropdown,
    ])

    action_box = widgets.HBox([preview_button, run_button, batch_button])
    patient_box = widgets.VBox([
        widgets.HTML('<b>Dane do zapisu DICOM</b>'),
        patient_first,
        patient_last,
        patient_pesel,
        patient_comment,
        dicom_button,
    ])

    root = widgets.VBox([params_box, action_box, patient_box, status_html, output])
    return root

In [12]:
# Quick start: uruchom GUI jednym kliknieciem

gui = build_gui()
display(gui)

## Uwagi
Interfejs GUI pokrywa logikę pierwotnego notatnika: podglad ukladu, sinogram, rekonstrukcja, filtracja, zapis DICOM i batch.